# 11 - Figures and tables for the paper

Every table and figure regenerates from saved predictions and SHAP values in one pass. Hand-assembled figures drift from the numbers in the text.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive; drive.mount('/content/drive')

REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}

import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd
from src.utils.manifest import load_manifest
mf = load_manifest(P['manifest'])
mf[['run_id','run_family','split_name','metrics.pr_auc','metrics.fpr_at_95_tpr','metrics.ece']]

In [ ]:
# Table 1: model comparison across all three splits
tab = (mf.pivot_table(index='run_family', columns='split_name',
                      values=['metrics.pr_auc','metrics.fpr_at_95_tpr'])
         .round(4))
tab.to_csv(f"{P['results']['tables']}/table1_model_comparison.csv")
tab.to_latex(f"{P['results']['tables']}/table1_model_comparison.tex")
tab

In [ ]:
# The random vs family-disjoint gap is itself a finding - report it explicitly
gap = (mf[mf.run_family=='fusion']
        .set_index('split_name')['metrics.pr_auc'])
print('optimism from random splitting:', round(gap.get('random_v1',0) - gap.get('family_disjoint_v1',0), 4))